# Mathematical Foundations of Logistic Regression

## Overview
This notebook dives deep into the mathematical concepts that power logistic regression. Understanding these foundations will help you better interpret the model's behavior and make informed decisions when applying it to real-world problems.

## Key Topics
1. Probability Theory Basics
2. Log Odds and Logit Function
3. Cost Function Derivation
4. Gradient Descent Optimization
5. Maximum Likelihood Estimation

Let's explore each concept in detail with mathematical derivations and visualizations.

In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification

# Set random seed for reproducibility
np.random.seed(42)

# Define sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## 1. Probability Theory Basics

Logistic regression is fundamentally based on probability theory. It models the probability of an instance belonging to a particular class.

For a binary classification problem:
- Let $P(y=1|X)$ be the probability of the positive class given features $X$
- Then $P(y=0|X) = 1 - P(y=1|X)$ is the probability of the negative class

Logistic regression assumes that this probability can be modeled using the sigmoid function:

$$P(y=1|X) = \sigma(w^TX + b) = \frac{1}{1 + e^{-(w^TX + b)}}$$

where $w$ is the weight vector and $b$ is the bias term.

## 2. Log Odds and Logit Function

The odds of an event is the ratio of the probability of the event occurring to the probability of it not occurring:

$$odds = \frac{P(y=1|X)}{P(y=0|X)} = \frac{P(y=1|X)}{1 - P(y=1|X)}$$

Taking the natural logarithm of the odds gives us the log-odds or logit:

$$logit(P) = log(\frac{P(y=1|X)}{1 - P(y=1|X)}) = w^TX + b$$

The logit function is the inverse of the sigmoid function. Let's visualize this relationship:

In [ ]:
def logit(p):
    return np.log(p / (1 - p))

# Generate points for visualization
p = np.linspace(0.01, 0.99, 100)
logit_values = logit(p)

plt.figure(figsize=(10, 6))
plt.plot(p, logit_values)
plt.title('Logit Function')
plt.xlabel('Probability (p)')
plt.ylabel('logit(p)')
plt.grid(True)
plt.show()

## 3. Cost Function Derivation

To train a logistic regression model, we need a cost function to minimize. We use the log-loss (also called binary cross-entropy) as our cost function.

For a single training example:

$$J(w,b) = -[y log(\hat{y}) + (1-y) log(1 - \hat{y})]$$

where:
- $y$ is the true label (0 or 1)
- $\hat{y} = \sigma(w^TX + b)$ is the predicted probability

For $m$ training examples, the average cost is:

$$J(w,b) = -\frac{1}{m} \sum_{i=1}^{m} [y^{(i)} log(\hat{y}^{(i)}) + (1-y^{(i)}) log(1 - \hat{y}^{(i)})]$$

This cost function is convex, meaning it has a single global minimum that we can reach using gradient descent.

## 4. Gradient Descent Optimization

To minimize the cost function, we use gradient descent. We iteratively update the parameters in the direction of steepest descent of the cost function.

The gradients for logistic regression are:

$$\frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)}) x_j^{(i)}$$

$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$$

Parameter update rules:

$$w_j := w_j - \alpha \frac{\partial J}{\partial w_j}$$

$$b := b - \alpha \frac{\partial J}{\partial b}$$

where $\alpha$ is the learning rate.

In [ ]:
# Implementation of logistic regression with gradient descent
class LogisticRegressionGD:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.weights = None
        self.bias = None
        self.costs = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Initialize parameters
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Gradient descent
        for _ in range(self.n_iterations):
            # Forward pass
            linear_model = np.dot(X, self.weights) + self.bias
            y_predicted = sigmoid(linear_model)
            
            # Compute cost
            cost = (-1/n_samples) * np.sum(y * np.log(y_predicted) + (1-y) * np.log(1-y_predicted))
            self.costs.append(cost)
            
            # Compute gradients
            dw = (1/n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1/n_samples) * np.sum(y_predicted - y)
            
            # Update parameters
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
    
    def predict_proba(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        return sigmoid(linear_model)
    
    def predict(self, X):
        return self.predict_proba(X) >= 0.5

Let's visualize how the cost decreases during gradient descent:

In [ ]:
# Generate synthetic dataset
X, y = make_classification(
    n_samples=100,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    random_state=42,
    n_clusters_per_class=1
)

# Train the model
model = LogisticRegressionGD(learning_rate=0.1, n_iterations=1000)
model.fit(X, y)

# Plot the cost over iterations
plt.figure(figsize=(10, 6))
plt.plot(model.costs)
plt.title('Cost Function During Gradient Descent')
plt.xlabel('Iteration')
plt.ylabel('Cost (Log Loss)')
plt.grid(True)
plt.show()

## 5. Maximum Likelihood Estimation

Logistic regression parameters can also be derived through Maximum Likelihood Estimation (MLE).

The likelihood function for $m$ training examples is:

$$L(w,b) = \prod_{i=1}^{m} P(y^{(i)}|x^{(i)};w,b)$$

Since dealing with products is difficult, we take the log-likelihood:

$$log L(w,b) = \sum_{i=1}^{m} log P(y^{(i)}|x^{(i)};w,b)$$

For binary classification:

$$log L(w,b) = \sum_{i=1}^{m} [y^{(i)} log(\hat{y}^{(i)}) + (1-y^{(i)}) log(1 - \hat{y}^{(i)})]$$

Notice that maximizing the log-likelihood is equivalent to minimizing the negative log-likelihood, which is exactly our cost function (log loss) from earlier.